# NóiLái — Kaggle 2×T4 evaluation runs

Runs one line of `configs/run_plan.yaml` (default `E1_main`) over a list of models from
`configs/models.yaml` with `scripts/run_eval.py --resume`, on Kaggle's free 2×T4 session
(12 h max, ~30 GPU-h/week; ≤ 9 h of work per session, outputs pushed after every model).
Models configured for other hardware (the TPU trio, the Modal L4) are skipped by the
session guard, never run on the wrong device. Every cell is non-interactive: set the
**parameters** cell, then *Run all*. Attach before starting:

* accelerator **GPU T4 ×2**; internet on;
* Kaggle Secrets: `GITHUB_TOKEN` (repository read, only if no bundle), `HF_TOKEN`
  (gated Gemma/Llama/Vistral), `KAGGLE_USERNAME` + `KAGGLE_KEY` (dataset push);
* a private dataset holding `noilai-main.bundle` (`git bundle create noilai-main.bundle main`)
  and one holding the frozen `data/release/` (never public: the test split is gated);
* from the second session on, the private runs dataset (`noilai-runs`, written by (g)) as an
  input, so that (b') restores `outputs.jsonl`, the manifests and the API ledger and `--resume`
  continues where the last session stopped instead of restarting every model at item 0.

Cells: (a) environment · (b) clone · (b') restore the persisted runs tree (`--resume` + ledger) ·
(c) pinned install (vLLM in a venv, DD 7.3) · (d) resources + item-file gate · (e) smoke test ·
(f) **RUN** · (g) outputs → dataset · (h) GPU-hours log.
The flag mapping to `run_eval.py` lives in `scripts/kaggle_run_plan.py`; adjust flags there
or through `EXTRA_ARGS`. Kaggle's own environment (nvidia driver, CUDA) is recorded by (c).

In [ ]:
# ---- parameters: edit this cell only; nothing below asks for input ------------------------------
import time
SESSION_T0 = time.time()

REPO_URL = "https://github.com/BEEEEEEBAGON/noilai.git"   # the split-out repository (docs/MIGRATION.md); BUNDLE_PATH wins when set
REPO_REF = "main"                                   # pin a commit hash for a paper run; recorded in every manifest
REPO_SUBDIR = "noilai"                              # "" once noilai is a repository of its own
BUNDLE_PATH = "/kaggle/input/noilai-bundle/noilai-main.bundle"   # git bundle in a private dataset; preferred over a token
CLONE_DIR = "/kaggle/working/repo"
WORK_DIR = "/kaggle/working"                        # ~20 GB, saved with the notebook
HF_HOME = "/kaggle/tmp/hf"                          # model cache outside the working dir [UNCERTAIN: verify the quota of /kaggle/tmp]
ITEMS_DATASET_DIR = "/kaggle/input/noilai-release"  # private dataset with the frozen data/release/<version>/ tree (or that tree's parent); None if the clone has it
RUNS_RESTORE_DIR = "/kaggle/input/noilai-runs"      # the private runs dataset (KAGGLE_DATASET_SLUG) attached as input: restored into data/runs by (b') so --resume and the ledger continue; None on the very first session
VERIFY_ITEM_KEYS = None          # None = derived from RUN_ID and the smoke run by (d); or list keys of configs/run_plan.yaml item_files

INSTALL_MODE = "venv"            # "venv": vLLM into /kaggle/tmp (DD 7.3: vLLM 0.30 pins torch 2.13, Kaggle ships 2.10); "kaggle_vllm": the Kaggle-built wheel; "system"
VENV_DIR = "/kaggle/tmp/vllm-venv"
VLLM_VERSION = "0.30.0"          # DESIGN_DECISIONS 7.3 [UNCERTAIN: verify] pin to the version the smoke tests pass with; recorded by (c)
KAGGLE_VLLM_VERSION = "0.2.0"    # INSTALL_MODE "kaggle_vllm" only [UNCERTAIN: verify]
TRANSFORMERS_VERSION = None      # None = whatever the vllm wheel requires (pip freeze is recorded either way)

GITHUB_TOKEN_SECRET = "GITHUB_TOKEN"
HF_TOKEN_SECRET = "HF_TOKEN"

RUN_ID = "E1_main"               # a run id from configs/run_plan.yaml (kaggle_run_plan.py --list)
MODELS = ["gemma-3-1b-it"]       # subset of the run's models; [] = every model of the run that this session's hardware can run
SMOKE_RUN_IDS = ["smoke_20"]     # (e) the smoke line, 20 items on SMOKE_MODEL before the run
SMOKE_MODEL = "gemma-3-1b-it"
EXTRA_ARGS = ""                  # extra run_eval.py flags, one shell-quoted string, e.g. "--seed 7"
ACCOUNT_HOLDER_ROLE = ""         # ROLE of the person holding the compute / hub accounts (e.g. "author", "adult collaborator"); never a name (DD 11.2); recorded in every manifest
ALLOW_UNPINNED_REVISION = False  # DD 7.1: a paper run needs `revision` pinned in configs/models.yaml; True only for a rehearsal, recorded in the manifest as null
PUSH_EVERY_MINUTES = 30          # push the outputs to the dataset this often WHILE a model runs (a killed session keeps the last push)
DRY_RUN = False                  # True prints the commands and runs nothing

PLATFORM, GPU_TYPE, N_GPUS = "kaggle", "t4", 2
SESSION_HARDWARE = "2xt4"        # models.yaml hardware key of THIS session; entries for other hardware are skipped (t4 entries run here)
ALLOW_HARDWARE_MISMATCH = False  # True runs e.g. a TPU-configured model here as the documented 2xT4 fallback (logged as this session's device)
MAX_SESSION_HOURS = 9.0          # DD 7.6: launch no new model past 9 h of a 12-h session
REQUIRE_GPU = True
RUN_LABEL = RUN_ID
PUSH_DATASET = True
PUSH_AFTER_EACH_MODEL = True     # DD 7.6: push mid-run so a timed-out session keeps what it finished
KAGGLE_DATASET_SLUG = "noilai-runs"
CREATE_DATASET = False           # True the first time the dataset is published

In [ ]:
# secrets: environment -> Colab userdata -> Kaggle Secrets. Values go into os.environ for the
# library that needs them and are reported as found / not found. Nothing here prints a value.
import os

def _secret(name):
    val = os.environ.get(name)
    if val:
        return val
    try:
        from google.colab import userdata          # Colab
        try:
            val = userdata.get(name)
        except Exception:
            val = None
        if val:
            return val
    except ImportError:
        pass
    try:
        from kaggle_secrets import UserSecretsClient   # Kaggle
        try:
            val = UserSecretsClient().get_secret(name)
        except Exception:
            val = None
        if val:
            return val
    except ImportError:
        pass
    return None

def _export(name, required=False):
    val = _secret(name)
    if val:
        os.environ[name] = val
        print(f"secret {name}: found ({len(val)} chars, not shown)")
        return True
    print(f"secret {name}: not found")
    if required:
        raise SystemExit(f"{name} is required for this notebook")
    return False

In [ ]:
# (a) environment check: interpreter, GPU(s), CUDA, vLLM's compute-capability floor (7.5), disk
import json, shutil, subprocess, sys
print(sys.version.split()[0], "on", sys.platform)
if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
else:
    print("nvidia-smi not found: is a GPU accelerator attached?")
ENV_CHECK = {"n_gpus": 0, "devices": [], "vllm_ok": False}
try:
    import torch
    print("torch", torch.__version__, "| cuda", torch.version.cuda, "| available", torch.cuda.is_available())
    for i in range(torch.cuda.device_count()):
        cap = torch.cuda.get_device_capability(i)
        mem = torch.cuda.get_device_properties(i).total_memory / 2**30
        ENV_CHECK["devices"].append({"name": torch.cuda.get_device_name(i), "capability": f"{cap[0]}.{cap[1]}", "memory_gb": round(mem, 1)})
        print(f"  gpu{i}: {torch.cuda.get_device_name(i)}  capability {cap[0]}.{cap[1]}  {mem:.1f} GB")
    ENV_CHECK["n_gpus"] = torch.cuda.device_count()
    ENV_CHECK["vllm_ok"] = ENV_CHECK["n_gpus"] > 0 and all(
        tuple(int(x) for x in d["capability"].split(".")) >= (7, 5) for d in ENV_CHECK["devices"])
except ImportError:
    print("torch is not importable yet (installed in the install cell)")
if REQUIRE_GPU and ENV_CHECK["n_gpus"] == 0:
    raise SystemExit("no CUDA device: choose the GPU accelerator (Kaggle: 'GPU T4 x2') and restart")
if ENV_CHECK["n_gpus"] and not ENV_CHECK["vllm_ok"]:
    print("WARNING: a device is below compute capability 7.5 (P100?) -> vLLM cannot run; use backend hf / llama.cpp")
du = shutil.disk_usage(WORK_DIR)
print(f"disk at {WORK_DIR}: {du.free / 2**30:.1f} GB free of {du.total / 2**30:.1f} GB")
print(json.dumps(ENV_CHECK))

In [ ]:
# (b) clone: from a git bundle in a private dataset / Drive when present (no token needed), else from
# GitHub with a token from the platform's secret store. The token reaches git only through GIT_ASKPASS,
# a two-line helper that echoes an environment variable: it is never in a URL, an argv or this output.
import os, stat, subprocess, sys
from pathlib import Path

clone_dir = Path(CLONE_DIR)
bundle = Path(BUNDLE_PATH) if BUNDLE_PATH else None
if bundle is not None and not bundle.exists():
    print(f"bundle {bundle} not found; falling back to {REPO_URL}")
    bundle = None
askpass = None
if bundle is None:
    if "OWNER/REPO" in REPO_URL:
        raise SystemExit("set REPO_URL (or provide BUNDLE_PATH) in the parameters cell")
    if _export(GITHUB_TOKEN_SECRET):
        askpass = Path.home() / ".noilai_askpass.sh"
        askpass.write_text("#!/bin/sh\necho \"$" + GITHUB_TOKEN_SECRET + "\"\n")
        askpass.chmod(stat.S_IRWXU)
# the environment git runs in is built AFTER the export, so the askpass helper finds the token in it
env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
if askpass is not None:
    env["GIT_ASKPASS"] = str(askpass)
source = str(bundle) if bundle else REPO_URL
if (clone_dir / ".git").exists():
    subprocess.run(["git", "-C", str(clone_dir), "fetch", "--tags", source, "+refs/heads/*:refs/remotes/origin/*"], check=True, env=env)
else:
    subprocess.run(["git", "clone", source, str(clone_dir)], check=True, env=env)
# a persisted clone may hold a STALE local branch named like REPO_REF: try the freshly fetched
# remote-tracking ref first and the literal ref last (commit hashes and tags have no remote form)
import re
cands = [REPO_REF] if re.fullmatch(r"[0-9a-f]{7,40}", REPO_REF) else [f"origin/{REPO_REF}", REPO_REF]
for cand in cands:
    if subprocess.run(["git", "-C", str(clone_dir), "checkout", "--detach", cand], env=env).returncode == 0:
        break
else:
    raise SystemExit(f"cannot check out {REPO_REF} (tried {cands})")
HEAD = subprocess.run(["git", "-C", str(clone_dir), "rev-parse", "HEAD"], check=True, env=env,
                      capture_output=True, text=True).stdout.strip()
PROJECT = clone_dir / REPO_SUBDIR if (clone_dir / REPO_SUBDIR / "pyproject.toml").exists() else clone_dir
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT / "scripts"))
print("HEAD", HEAD)
print("project", PROJECT)

In [ ]:
# (b') restore: the clone is ephemeral, so the runs tree persisted by the previous session (the private
# Kaggle dataset `noilai-runs` attached as input, or the Drive folder the API notebook copies to) is
# copied back into PROJECT/data/runs BEFORE anything runs: outputs.jsonl / manifest.json / unchanged.jsonl
# per run directory (so `--resume` continues at the first undone row instead of item 0), api_ledger.json
# (so a day's quota is not re-spent on rows already answered; DD 7.3 "a hit never re-calls") and the
# compute log (so the session's rows append to the campaign's). Nothing here overwrites a newer local file.
import shutil
from pathlib import Path
RESTORED = {"from": None, "run_dirs": 0, "ledger": False, "compute_log": False}
_src = Path(RUNS_RESTORE_DIR) if RUNS_RESTORE_DIR else None
if _src is not None and _src.exists():
    runs_src = _src / "runs" if (_src / "runs").is_dir() else _src        # the dataset holds runs/ + compute_log.csv, or the runs tree itself
    runs_dst = PROJECT / "data" / "runs"
    runs_dst.mkdir(parents=True, exist_ok=True)
    for child in sorted(runs_src.iterdir()):
        if child.name in ("__pycache__", ".ipynb_checkpoints"):
            continue
        if child.is_dir():
            shutil.copytree(child, runs_dst / child.name, dirs_exist_ok=True)
            RESTORED["run_dirs"] += 1
        elif child.name == "api_ledger.json":
            shutil.copy2(child, runs_dst / child.name)
            RESTORED["ledger"] = True
        elif child.suffix in (".json", ".jsonl", ".txt", ".csv"):
            shutil.copy2(child, runs_dst / child.name)
    for cand in (_src / "compute_log.csv", runs_src / "compute_log.csv"):
        if cand.exists() and not (PROJECT / "data" / "compute_log.csv").exists():
            shutil.copy2(cand, PROJECT / "data" / "compute_log.csv")
            RESTORED["compute_log"] = True
            break
    RESTORED["from"] = str(runs_src)
    print("restored", RESTORED)
else:
    print("nothing to restore (first session, or RUNS_RESTORE_DIR", RUNS_RESTORE_DIR, "is not attached): every run starts at item 0")

In [ ]:
# (c) pinned install per DESIGN_DECISIONS 7.3. vLLM 0.30.0 pins torch 2.13 while Kaggle ships torch
# 2.10 / CUDA 12.8, so the engine goes into its own venv under /kaggle/tmp (INSTALL_MODE "venv"),
# or the Kaggle-built wheel is used (INSTALL_MODE "kaggle_vllm"); "system" installs into the kernel's
# interpreter and is for a runtime that already matches. The project (eval extra) is installed into
# the same interpreter, RUN_PYTHON, which every run_eval.py process is launched with; pip freeze and
# the framework versions of THAT interpreter are recorded next to the outputs (checklist C4).
import subprocess, sys
from pathlib import Path
if INSTALL_MODE == "venv":
    venv = Path(VENV_DIR)
    if not (venv / "bin" / "python").exists():
        subprocess.run([sys.executable, "-m", "venv", str(venv)], check=True)
    RUN_PYTHON = str(venv / "bin" / "python")
    subprocess.run([RUN_PYTHON, "-m", "pip", "install", "-q", "--upgrade", "pip"], check=True)
    pins = [f"vllm=={VLLM_VERSION}"]
elif INSTALL_MODE == "kaggle_vllm":
    RUN_PYTHON = sys.executable
    pins = [f"kaggle-vllm=={KAGGLE_VLLM_VERSION}"]
elif INSTALL_MODE == "system":
    RUN_PYTHON = sys.executable
    pins = [f"vllm=={VLLM_VERSION}"]
else:
    raise SystemExit(f"INSTALL_MODE must be venv, kaggle_vllm or system, not {INSTALL_MODE!r}")
if TRANSFORMERS_VERSION:
    pins.append(f"transformers=={TRANSFORMERS_VERSION}")
subprocess.run([RUN_PYTHON, "-m", "pip", "install", "-q", *pins], check=True)      # the engine first: its requirements win
subprocess.run([RUN_PYTHON, "-m", "pip", "install", "-q", "-e", ".[eval]"], check=True)
if RUN_PYTHON != sys.executable:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)   # the driver imports the project too
os.environ["HF_HOME"] = HF_HOME
Path(HF_HOME).mkdir(parents=True, exist_ok=True)
_export(HF_TOKEN_SECRET)          # gated checkpoints (Gemma, Llama, Vistral); exported as HF_TOKEN when the secret has that name
if HF_TOKEN_SECRET != "HF_TOKEN" and os.environ.get(HF_TOKEN_SECRET):
    os.environ["HF_TOKEN"] = os.environ[HF_TOKEN_SECRET]
import colab_setup as CS
ENV_RECORD = CS.record_environment(PROJECT / "data" / "runs" / "env", python=RUN_PYTHON)
print("engine interpreter:", RUN_PYTHON)
print("environment recorded at", ENV_RECORD)

In [ ]:
# (d) third-party resources (SHA-256 against data/HASHES.json) and the item file(s) of the runs this
# notebook will execute: the keys are DERIVED from the run ids (plus the smoke run) unless
# VERIFY_ITEM_KEYS lists them; a seeded sub-sample the plan derives from another file is written first
# (--materialize). Each file is checked for its hash against configs/run_plan.yaml, the header record,
# the canary on every row and its row/cell counts. A failure stops the notebook here.
import shutil, subprocess, sys
from pathlib import Path
import kaggle_run_plan as KRP
r = subprocess.run([sys.executable, "scripts/fetch_resources.py"], check=False)
if r.returncode:
    raise SystemExit("resource fetch or hash check failed (see above)")
if ITEMS_DATASET_DIR:
    src = Path(ITEMS_DATASET_DIR)
    if not src.exists():
        raise SystemExit(f"ITEMS_DATASET_DIR {src} not found: attach the private dataset that holds data/release")
    rel = Path(KRP.load_plan()["release"])            # e.g. data/release/v0.3: the plan's `release` key decides where the files go
    dest = PROJECT / rel
    if (src / rel.name).exists():                      # the dataset holds the data/release tree (one directory per version)
        shutil.copytree(src / rel.name, dest, dirs_exist_ok=True)
    elif (src / "manifest.json").exists():             # the dataset holds ONE release directory
        shutil.copytree(src, dest, dirs_exist_ok=True)
    else:
        raise SystemExit(f"{src} holds neither {rel.name}/ nor a release manifest.json: attach the frozen release")
    print("release copied from", src, "to", dest)
_run_ids = list(globals().get("RUN_IDS") or [globals().get("RUN_ID")]) + list(globals().get("SMOKE_RUN_IDS") or [])
KEYS = VERIFY_ITEM_KEYS or KRP.item_keys_for_runs(KRP.load_plan(), [r for r in _run_ids if r])
print("item files to verify:", KEYS)
for key in KEYS:
    r = subprocess.run([sys.executable, "scripts/kaggle_verify_items.py", "--key", key, "--materialize"], check=False)
    if r.returncode:
        raise SystemExit(f"item file {key} failed verification; nothing runs on it")

In [ ]:
# outputs hand-off used after EVERY model (DD 7.6: Kaggle keeps /kaggle/working only if the version
# completes, so a session that times out must already have pushed what it finished) and once more at
# the end: copy data/runs (+ compute log) into /kaggle/working and add a version to a private Kaggle
# dataset through scripts/kaggle_dataset.py (credentials from Secrets, never printed).
import os, shutil, subprocess, sys
from pathlib import Path

def push_outputs(label):
    out = Path(WORK_DIR) / "noilai_runs_out"
    shutil.copytree(PROJECT / "data" / "runs", out / "runs", dirs_exist_ok=True)
    if (PROJECT / "data" / "compute_log.csv").exists():
        shutil.copy2(PROJECT / "data" / "compute_log.csv", out / "compute_log.csv")
    print("outputs copied to", out)
    if not PUSH_DATASET:
        return
    for name in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
        _export(name)
    if os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"):
        cmd = [sys.executable, "scripts/kaggle_dataset.py", "--src", str(PROJECT / "data" / "runs"),
               "--src", str(PROJECT / "data" / "compute_log.csv"), "--dest", str(Path(WORK_DIR) / "noilai_runs_dataset"),
               "--slug", KAGGLE_DATASET_SLUG, "--message", f"{label} @ {HEAD[:8]}"]
        if CREATE_DATASET and not globals().get("_DATASET_CREATED"):
            cmd.append("--create")
        rc = subprocess.run(cmd, check=False).returncode
        if rc == 0:
            globals()["_DATASET_CREATED"] = True
    else:
        print("Kaggle credentials not found: dataset push skipped (outputs are still under", out, ")")

def after_each_model(res):
    if PUSH_AFTER_EACH_MODEL:
        push_outputs(f"{res['run']}/{res['model']} {res['status']}")

In [ ]:
# after each RUN cell: scripts/check_run.py on every finished run directory: item gate, deterministic rescoring,
# stats.json (accuracy with the base-pair cluster-bootstrap CI, the copy-baseline gain of PREREG section 10, the paired
# NFD contrast, the EXPLORATORY T1 forced choice against chance) and results_hashes.json
import json, subprocess, sys

def check_runs(results):
    summaries = []
    for r in results:
        run_dir = PROJECT / "data" / "runs" / f"{r['run']}__{r['model']}"
        if not (run_dir / "scores.jsonl").exists():
            print(r["run"], r["model"], "->", r["status"], "(no scores: not checked)")
            continue
        c = subprocess.run([sys.executable, "scripts/check_run.py", "--run", str(run_dir)], capture_output=True, text=True, check=False)
        print(c.stdout.strip() or c.stderr[-1500:])
        st = json.loads((run_dir / "stats.json").read_text()) if (run_dir / "stats.json").exists() else {}
        summaries.append({"run": r["run"], "model": r["model"], "status": r["status"],
                          "by_task_arm": {k: (round(v["accuracy"], 3), [round(x, 3) for x in v["ci"]]) for k, v in st.get("by_task_arm", {}).items()},
                          "t1_forced_choice": {k: (round(v["accuracy"], 3), round(v["chance"], 3)) for k, v in st.get("t1_forced_choice", {}).items()}})
    print(json.dumps(summaries, ensure_ascii=False, indent=1))
    return summaries

### RUN CELL (e) — smoke test
One small model, 20 items (`smoke_20` in `run_plan.yaml`, the only kind of line allowed a
`--limit`): load, generate, log-probs. Stops the notebook on failure so that no GPU hour is
spent on a broken environment.

In [ ]:
# RUN CELL (e): smoke test. Flags come from run_plan.yaml (smoke_20) via scripts/kaggle_run_plan.py;
# the process runs on the engine's interpreter (RUN_PYTHON) and the session guards apply.
import shlex
import kaggle_run_plan as KRP
RUN_EXTRA = shlex.split(EXTRA_ARGS) + (["--account-holder", ACCOUNT_HOLDER_ROLE] if ACCOUNT_HOLDER_ROLE else [])   # role only, never a name (DD 11.2)
if not ACCOUNT_HOLDER_ROLE:
    print("WARNING: ACCOUNT_HOLDER_ROLE is empty; the manifests will record account_holder 'unknown' (DD 11.2 asks for the role)")
SMOKE_RESULTS = []
for smoke_id in SMOKE_RUN_IDS:
    SMOKE_RESULTS += KRP.execute(smoke_id, models=[SMOKE_MODEL], platform=PLATFORM, dry_run=DRY_RUN,
                                 continue_on_error=False, extra=RUN_EXTRA, python=RUN_PYTHON,
                                 session_hardware=SESSION_HARDWARE, allow_hardware_mismatch=ALLOW_HARDWARE_MISMATCH,
                                 session_t0=SESSION_T0, max_session_hours=MAX_SESSION_HOURS)
for r in SMOKE_RESULTS:
    print(r["run"], r["model"], "->", r["status"])
if any(str(r["status"]).startswith(("failed", "skipped", "refused")) for r in SMOKE_RESULTS):
    raise SystemExit("smoke test failed or was skipped: fix the environment, the model entry or SESSION_HARDWARE before the run cell")

### RUN CELL (f) — the run
Loops over `MODELS` (or every model of `RUN_ID` that `SESSION_HARDWARE` can run) calling
`scripts/run_eval.py --model-config <name> --items ... --tasks ... --arms ... --resume --run-id <RUN_ID>__<name> --out-root data/runs`
on the engine's interpreter. `--resume` makes a re-run after a session timeout continue where it
stopped — provided (b') restored the previous session's `data/runs` from the runs dataset; no new
model starts past `MAX_SESSION_HOURS`; the outputs are pushed every `PUSH_EVERY_MINUTES` while a
model runs and after every model; a self-hosted model whose `revision` is unpinned is skipped
(DD 7.1) unless `ALLOW_UNPINNED_REVISION`. Each model's wall time is appended to
`data/compute_log.csv` (a provisional 0-hour row at its start, the final row at its end) with THIS
session's device type and count (2 × T4), whatever hardware the model is configured for.

In [ ]:
# RUN CELL (f): the run matrix line RUN_ID over MODELS, with --resume. Adjust flags in
# scripts/kaggle_run_plan.py (build_command) or through EXTRA_ARGS; the CLI is owned by noilai/eval.
# A background thread pushes the outputs every PUSH_EVERY_MINUTES while a model runs (outputs.jsonl is
# flushed per row, so the pushed copy is resumable); the driver writes a provisional 0-hour compute-log
# row at each model's start, so a session killed mid-model leaves its start on record.
import json, threading
import kaggle_run_plan as KRP
_stop_push = threading.Event()

def _periodic_push():
    while not _stop_push.wait(PUSH_EVERY_MINUTES * 60):
        try:
            push_outputs("periodic")
        except Exception as e:      # a failed periodic push never stops the run
            print("periodic push failed:", repr(e))
_pusher = threading.Thread(target=_periodic_push, daemon=True)
_pusher.start()
try:
    RESULTS = KRP.execute(RUN_ID, models=MODELS or None, platform=PLATFORM, dry_run=DRY_RUN, continue_on_error=True,
                          extra=RUN_EXTRA, python=RUN_PYTHON, session_hardware=SESSION_HARDWARE,
                          allow_hardware_mismatch=ALLOW_HARDWARE_MISMATCH, session_t0=SESSION_T0,
                          max_session_hours=MAX_SESSION_HOURS, after_each=after_each_model,
                          allow_unpinned_revision=ALLOW_UNPINNED_REVISION,
                          line_overrides=globals().get("LINE_OVERRIDES"), chunk_tag=globals().get("CHUNK_TAG"))
finally:
    _stop_push.set()
    _pusher.join(timeout=5)
print(json.dumps([{"model": r["model"], "status": r["status"], "hours": r.get("hours"),
                   "new_outputs": r.get("new_outputs"), "logged_device": r.get("logged_device")} for r in RESULTS], indent=1))

In [ ]:
# (f') hashed results: scripts/check_run.py on every run directory this session wrote (scores it first if needed; item gate,
# deterministic rescoring, stats.json, results_hashes.json); the copies pushed in (g) carry them
CHECKED = check_runs(list(globals().get("SMOKE_RESULTS", [])) + list(globals().get("RESULTS", [])))

In [ ]:
# (g) final hand-off: the same push as after each model, once more for the session as a whole.
push_outputs(RUN_LABEL)

In [ ]:
# (h) GPU-hours log (checklist C1). kaggle_run_plan wrote one row per executed model run; this adds the
# session's remaining wall time (startup, install, model downloads) as "overhead" and prints the totals.
import shutil
import compute_log as CL
LOG = PROJECT / "data" / "compute_log.csv"
run_hours = sum((r.get("hours") or 0.0) for r in (globals().get("RESULTS", []) + globals().get("SMOKE_RESULTS", [])))
overhead = max(0.0, CL.hours_since(SESSION_T0) - run_hours)
CL.append_entry(LOG, CL.Entry(date=CL.today(), platform=PLATFORM, gpu_type=GPU_TYPE, n_gpus=N_GPUS,
                              hours=overhead, run_id=f"session-{RUN_LABEL}", purpose="overhead"))
print(CL.format_totals(CL.totals(CL.read_entries(LOG))))
out = Path(WORK_DIR) / "noilai_runs_out"
out.mkdir(parents=True, exist_ok=True)
shutil.copy2(LOG, out / "compute_log.csv")
print("log copied to", out / "compute_log.csv", "-> merge into the repository's data/compute_log.csv")